# HYTA — Segmentasi Awan/Langit (`05c`)

Notebook eksperimen **HYTA (*Hybrid Thresholding Algorithm*)** untuk deteksi awan pada
citra langit RGB. Dua tahap (lihat `docs/pelatihan/br_ratio_normalization/BR_RATIO_NORMALIZATION.md`):

1. **Tahap 1 — Normalisasi Rasio B/R** → peta `λ_N = (B − R) / (B + R + ε)`.
2. **Tahap 2 — Thresholding Hibrida** → `std(λ_N) > Ts` ⇒ **bimodal** (MCE), selain itu
   **unimodal** (fixed `Tf`). Piksel `λ_N < ambang` = **AWAN**, selainnya **LANGIT**.

**Keluaran:** peta biner awan/langit per citra + ringkasan statistik.

> **PENTING — jangan auto-run.** Notebook sengaja **belum dijalankan** (tanpa output).
> Jalankan sel dari atas ke bawah saat siap. MCE diimplementasikan sendiri (self-contained),
> tidak mengubah kode `src/`.

## 1. Setup & path repo

In [1]:
import os, sys, json, csv
from pathlib import Path

REPO = Path.cwd()
for _ in range(6):
    if (REPO / 'src').exists():
        break
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

print('repo :', REPO)
print('torch:', torch.__version__)

repo : d:\n8n-logsiswaparalayang\Jurnal\CitraAwan
torch: 2.1.1+cu121


## 2. Konfigurasi HYTA

Parameter dari makalah (Li, Lu & Yang 2011). `Ts`/`Tf` mungkin perlu dikalibrasi ulang
untuk dataset CitraAwan.

In [2]:
# --- sumber data (train seimbang combined_balanced; sama dengan baseline lain) ---
TRAIN_DIR = REPO / 'data' / 'combined_balanced' / 'train'
TRAIN_MANIFEST = REPO / 'reports' / 'tables' / 'combined_balanced_train_manifest.csv'
CLASS_NAMES = ['Cu', 'Ac', 'Ci', 'Cl', 'Sc', 'Cb', 'Mi']  # label 0..6 (configs/base.yaml)

# --- parameter HYTA ---
IMAGE_SIZE = 512
EPS = 1e-6          # pengaman bagi-nol pada (b + r + eps)  [sesuai pseudocode HYTA]
TS = 0.03           # ambang std-dev pemisah unimodal/bimodal
TF = 0.250          # ambang tetap (fixed) untuk kasus unimodal
MCE_NBINS = 256     # jumlah bin histogram untuk MCE
N_SAMPLES = 7       # 1 citra per kelas (tampilkan semua 7 kelas bila tersedia)

# --- output ARTEFAK -> experiments/ (netral, lepas dari densenet) ---
OUT_DIR = REPO / 'experiments' / 'exp_hyta_segmentation'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_PNG = OUT_DIR / 'hyta_samples_preview.png'
MASKS_PT = OUT_DIR / 'hyta_masks.pt'
CONFIG_JSON = OUT_DIR / 'hyta_config.json'
STATS_CSV = OUT_DIR / 'hyta_per_sample_stats.csv'
print('TRAIN_DIR exists :', TRAIN_DIR.exists())
print('MANIFEST exists  :', TRAIN_MANIFEST.exists())
print('OUT_DIR          :', OUT_DIR)

TRAIN_DIR exists : True
MANIFEST exists  : True
OUT_DIR          : d:\n8n-logsiswaparalayang\Jurnal\CitraAwan\experiments\exp_hyta_segmentation


## 3. HYTA — Tahap 1 (B/R) & Tahap 2 (thresholding hibrida)

`mce_threshold` mengimplementasikan **Minimum Cross Entropy** (Li & Lee) pada histogram
`λ_N`: intensitas-proksi = indeks bin `1..nbins` (positif, monoton terhadap `λ_N`), lalu
cari ambang yang meminimalkan cross-entropy. Nilai ambang dikonversi balik ke skala `λ_N`.

In [3]:
def br_ratio_lambda(x, eps=EPS):
    """Tahap 1: peta rasio B/R ternormalisasi. x: 3xHxW RGB [0,1] (0=R,1=G,2=B)."""
    assert x.ndim == 3 and x.shape[0] == 3, 'harap tensor RGB 3xHxW'
    r = x[0]
    b = x[2]
    lam = (b - r) / (b + r + eps)      # HxW, rentang teoretis (-1, 1)
    return lam


def mce_threshold(values, nbins=MCE_NBINS, vmin=None, vmax=None):
    """Minimum Cross Entropy threshold (Li & Lee) atas histogram `values`.

    Mengembalikan nilai ambang pada skala `values` (bukan indeks bin).
    """
    v = np.asarray(values, dtype=np.float64).ravel()
    if vmin is None:
        vmin = float(v.min())
    if vmax is None:
        vmax = float(v.max())
    if vmax <= vmin:
        return vmin
    hist, edges = np.histogram(v, bins=nbins, range=(vmin, vmax))
    h = hist.astype(np.float64)
    k = np.arange(1, nbins + 1, dtype=np.float64)   # intensitas-proksi positif
    kh = k * h
    best_t, best_eta = 1, np.inf
    for t in range(1, nbins):                       # pisah antara bin t dan t+1
        w1 = h[:t].sum()
        w2 = h[t:].sum()
        if w1 == 0 or w2 == 0:
            continue
        m1 = kh[:t].sum() / w1
        m2 = kh[t:].sum() / w2
        # eta = sum k*h*log(k/m); konstanta sum(k*h*log k) diabaikan (tak bergantung t)
        eta = -(kh[:t] * np.log(m1)).sum() - (kh[t:] * np.log(m2)).sum()
        if eta < best_eta:
            best_eta, best_t = eta, t
    return float(edges[best_t])                     # batas kanan bin optimal


def hyta_segment(x, eps=EPS, ts=TS, tf=TF, nbins=MCE_NBINS):
    """HYTA penuh. Mengembalikan (lambda_N HxW np, mask_awan HxW bool, meta)."""
    lam = br_ratio_lambda(x, eps).detach().cpu().numpy()
    std = float(lam.std())
    if std > ts:
        mode, thr = 'bimodal', mce_threshold(lam, nbins=nbins)
    else:
        mode, thr = 'unimodal', float(tf)
    mask_cloud = lam < thr                           # True = AWAN
    return lam, mask_cloud, {'mode': mode, 'threshold': thr, 'std': std,
                             'cloud_fraction': float(mask_cloud.mean())}

# pemuatan citra: PIL -> resize -> ToTensor [0,1]
load_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

def load_rgb01(path):
    with Image.open(path) as im:
        return load_tf(im.convert('RGB'))

## 4. Ambil sampel (1 per kelas)

Memakai pembaca manifes proyek (`src.datasets.manifest.load_samples`) agar resolusi path
konsisten dengan pipeline pelatihan.

In [4]:
from src.datasets.manifest import load_samples

samples = load_samples(TRAIN_MANIFEST, TRAIN_DIR)

picked, seen = [], set()
for s in samples:
    if s.label not in seen:
        picked.append(s)
        seen.add(s.label)
    if len(picked) >= N_SAMPLES:
        break

print(f'total sampel manifes: {len(samples)}')
for s in picked:
    print(f'  label {s.label} ({CLASS_NAMES[s.label]}):', s.path.name)

total sampel manifes: 21000
  label 0 (Cu): 1_cumulus_000002.jpg
  label 1 (Ac): 2_altocumulus_000001.jpg
  label 2 (Ci): 3_cirrus_000003.jpg
  label 3 (Cl): 4_clearsky_000001.jpg
  label 4 (Sc): 5_stratocumulus_000001.jpg
  label 5 (Cb): 6_cumulonimbus_000001.jpg
  label 6 (Mi): 7_mixed_000001.jpg


## 5. Jalankan HYTA & tampilkan

Baris = sampel; kolom = `asli (RGB)` | `λ_N` | `mask awan (putih)` | `overlay`.
Judul tiap baris mencantumkan mode (unimodal/bimodal), ambang, std, dan fraksi awan.

In [5]:
rows = len(picked)
fig, axes = plt.subplots(rows, 4, figsize=(12, 3 * rows))
if rows == 1:
    axes = axes[None, :]

records, mask_list, lam_list = [], [], []
for i, s in enumerate(picked):
    rgb = load_rgb01(s.path)                      # 3xHxW [0,1]
    lam, mask, meta = hyta_segment(rgb)
    mask_list.append(torch.from_numpy(mask))
    lam_list.append(torch.from_numpy(lam))
    records.append({'label': s.label, 'class': CLASS_NAMES[s.label], 'file': s.path.name,
                    **meta})

    axes[i, 0].imshow(rgb.permute(1, 2, 0).numpy())
    axes[i, 0].set_ylabel(f'{s.label} · {CLASS_NAMES[s.label]}', fontsize=10)
    axes[i, 0].set_title('asli (RGB)' if i == 0 else '')

    axes[i, 1].imshow(lam, cmap='coolwarm', vmin=-1, vmax=1)
    axes[i, 1].set_title('λ_N = (B-R)/(B+R+ε)' if i == 0 else '')

    axes[i, 2].imshow(mask, cmap='gray')
    axes[i, 2].set_title('mask AWAN (putih)' if i == 0 else '')

    ov = rgb.permute(1, 2, 0).numpy().copy()
    ov[mask] = 0.5 * ov[mask] + 0.5 * np.array([1.0, 0.0, 0.0])  # awan -> merah
    axes[i, 3].imshow(ov)
    axes[i, 3].set_title('overlay awan' if i == 0 else '')
    axes[i, 3].set_xlabel(f"{meta['mode']} · t={meta['threshold']:.3f} · "
                          f"std={meta['std']:.3f} · awan={meta['cloud_fraction']:.2f}",
                          fontsize=8)

    for j in range(4):
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])

fig.suptitle('HYTA — segmentasi awan/langit', y=1.0)
fig.tight_layout()
fig.savefig(PREVIEW_PNG, dpi=120, bbox_inches='tight')
print('tersimpan:', PREVIEW_PNG)
plt.show()

tersimpan: d:\n8n-logsiswaparalayang\Jurnal\CitraAwan\experiments\exp_hyta_segmentation\hyta_samples_preview.png


<Figure size 1200x2100 with 28 Axes>

## 6. Ringkasan statistik per sampel

In [6]:
n_bimodal = sum(1 for r in records if r['mode'] == 'bimodal')
print(f"bimodal (MCE): {n_bimodal} | unimodal (fixed Tf): {len(records) - n_bimodal}")
for r in records:
    print(f"  {r['label']} {r['class']:>3} | {r['mode']:>8} | t={r['threshold']:.3f} "
          f"| std={r['std']:.3f} | fraksi_awan={r['cloud_fraction']:.3f} | {r['file']}")

bimodal (MCE): 2 | unimodal (fixed Tf): 5
  0  Cu |  bimodal | t=0.342 | std=0.214 | fraksi_awan=0.388 | 1_cumulus_000002.jpg
  1  Ac |  bimodal | t=0.539 | std=0.226 | fraksi_awan=0.514 | 2_altocumulus_000001.jpg
  2  Ci | unimodal | t=0.250 | std=0.022 | fraksi_awan=1.000 | 3_cirrus_000003.jpg
  3  Cl | unimodal | t=0.250 | std=0.021 | fraksi_awan=0.006 | 4_clearsky_000001.jpg
  4  Sc | unimodal | t=0.250 | std=0.011 | fraksi_awan=1.000 | 5_stratocumulus_000001.jpg
  5  Cb | unimodal | t=0.250 | std=0.017 | fraksi_awan=1.000 | 6_cumulonimbus_000001.jpg
  6  Mi | unimodal | t=0.250 | std=0.028 | fraksi_awan=1.000 | 7_mixed_000001.jpg


## 7. Simpan artefak

Menyimpan mask + peta `λ_N` + metadata ke `experiments/exp_hyta_segmentation/`.

In [7]:
masks = torch.stack(mask_list, dim=0)          # [N, H, W] bool
lams = torch.stack(lam_list, dim=0)            # [N, H, W] float
labels = torch.tensor([s.label for s in picked])

torch.save({
    'masks_cloud': masks,
    'lambda_N': lams,
    'labels': labels,
    'class_names': CLASS_NAMES,
    'paths': [str(s.path) for s in picked],
    'params': {'eps': EPS, 'Ts': TS, 'Tf': TF, 'mce_nbins': MCE_NBINS},
}, MASKS_PT)

cfg = {
    'algorithm': 'HYTA (Hybrid Thresholding Algorithm)',
    'stage1': 'lambda_N = (B - R) / (B + R + eps)',
    'stage2': 'std(lambda_N) > Ts ? MCE : fixed Tf ; cloud = lambda_N < threshold',
    'eps': EPS, 'Ts': TS, 'Tf': TF, 'mce_nbins': MCE_NBINS,
    'image_size': IMAGE_SIZE, 'channel_order': 'RGB (0=R,1=G,2=B)',
    'reference': 'Li, Lu & Yang (2011), JTECH-D-11-00009.1',
    'doc': 'docs/pelatihan/br_ratio_normalization/BR_RATIO_NORMALIZATION.md',
}
with open(CONFIG_JSON, 'w', encoding='utf-8') as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)

with open(STATS_CSV, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['label', 'class', 'file', 'mode', 'threshold',
                                      'std', 'cloud_fraction'])
    w.writeheader()
    for r in records:
        w.writerow(r)

print('tersimpan:', MASKS_PT)
print('tersimpan:', CONFIG_JSON)
print('tersimpan:', STATS_CSV)
print('masks:', tuple(masks.shape), '| lambda_N:', tuple(lams.shape))

tersimpan: d:\n8n-logsiswaparalayang\Jurnal\CitraAwan\experiments\exp_hyta_segmentation\hyta_masks.pt
tersimpan: d:\n8n-logsiswaparalayang\Jurnal\CitraAwan\experiments\exp_hyta_segmentation\hyta_config.json
tersimpan: d:\n8n-logsiswaparalayang\Jurnal\CitraAwan\experiments\exp_hyta_segmentation\hyta_per_sample_stats.csv
masks: (7, 512, 512) | lambda_N: (7, 512, 512)


## 8. Catatan & langkah lanjut

- **MCE** diimplementasikan via histogram (Li & Lee); bandingkan dengan `skimage.filters.threshold_li`
  bila perlu verifikasi.
- **Integrasi ke klasifikasi 7-kelas belum diputuskan** (fitur/mask tambahan vs masking vs
  baseline segmentasi) — lihat `BR_RATIO_NORMALIZATION.md` §5.
- Kalibrasi `Ts=0.03`, `Tf=0.250` untuk dataset CitraAwan; kanal hijau `G` tidak dipakai pada `λ_N`.
- Bila dipromosikan, pindahkan fungsi HYTA ke `src/` (mis. `src/preprocessing/hyta.py`).